# Fertilizer RAS — all three nutrients (N, P, K)

Runs the full pipeline for N, P and K independently and reports a
cross-nutrient summary (production loss, trade change, coverage).

In [ ]:
import sys, pathlib

sys.path.append(str(pathlib.Path.cwd().parent))

import pandas as pd

from src.preprocessing import (
    load_inputs, load_trade,
    get_production_demand, get_trade_matrix,
    filter_countries, apply_shock, NUTRIENT_NAMES,
)
from src.model import FertilizerRAS
from src.postprocessing import build_comparison, global_summary, save_result

pd.set_option('display.float_format', '{:,.0f}'.format)

In [ ]:
YEARS = list(range(2014, 2019))
MIN_THRESHOLD = 1_000
INPUTS_CSV = '../data/Inputs_FertilizersNutrient_E_All_Data/Inputs_FertilizersNutrient_E_All_Data_NOFLAG.csv'
TRADE_CSV  = '../data/Fertilizers_DetailedTradeMatrix_E_All_Data/Fertilizers_DetailedTradeMatrix_E_All_Data_NOFLAG.csv'
SHOCK = {
    'Russian Federation': 0.40,
    'Belarus':            0.30,
    'Ukraine':            0.20,
}

In [ ]:
inputs_df = load_inputs(INPUTS_CSV, YEARS)
trade_df  = load_trade(TRADE_CSV,  YEARS)

all_results = {}
for nut in ['N', 'P', 'K']:
    P_raw, C_raw = get_production_demand(inputs_df, nut)
    T0_raw       = get_trade_matrix(trade_df, nut)
    _, P, C, T0 = filter_countries(P_raw, C_raw, T0_raw, min_threshold=MIN_THRESHOLD)

    P_shocked = apply_shock(P, SHOCK)
    baseline  = FertilizerRAS(P,         C, T0).run()
    shocked   = FertilizerRAS(P_shocked, C, T0).run()
    comparison = build_comparison(baseline, shocked)

    all_results[nut] = dict(baseline=baseline, shocked=shocked, comparison=comparison)

    save_result(baseline, '../results', tag=f'{nut}_baseline')
    save_result(shocked,  '../results', tag=f'{nut}_shocked')

    print(f'\n=== {NUTRIENT_NAMES[nut]} ===')
    display(global_summary(baseline, shocked))

In [ ]:
# Cross-nutrient summary
rows = []
for nut, r in all_results.items():
    b, s = r['baseline'], r['shocked']
    rows.append({
        'Nutrient':      NUTRIENT_NAMES[nut],
        'Countries':     len(b.P),
        'Prod_loss':     float(s.P.sum() - b.P.sum()),
        'Trade_delta':   float(s.X.values.sum() - b.X.values.sum()),
        'Cover_base_%':  float(b.F.sum() / max(b.C.sum(), 1e-12) * 100),
        'Cover_shock_%': float(s.F.sum() / max(s.C.sum(), 1e-12) * 100),
    })
cross = pd.DataFrame(rows).set_index('Nutrient').round(2)
display(cross)